# PreDist 데이터셋 확인 노트북

이 노트북은 `05_데이터셋/PreDist`에 있는 PreDist 원본 zip과 metadata CSV를 pandas DataFrame으로 열어 구조를 확인한다.

실행 기준:

```powershell
uv venv
uv sync
uv run python -c "import pandas, ipykernel, nbformat, nbclient; print('ok')"
```

`jupyter lab`은 사용하지 않는다. 노트북 검증은 `nbclient`로 현재 작업 폴더에서 전체 셀을 실행한다.

## 1. 데이터 불러오기

### 1.1 라이브러리 import

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import TextIOWrapper

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)

pandas 3.0.3


### 1.2 데이터 경로 설정

노트북을 프로젝트 루트 또는 `06_노트북`에서 실행해도 같은 경로를 찾도록 처리한다.

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "05_데이터셋").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_ROOT / "predist_dataset.zip"
METADATA_DIR = DATA_ROOT / "metadata"

assert PROJECT_ROOT.exists(), f"프로젝트 루트가 없습니다: {PROJECT_ROOT}"
assert ZIP_PATH.exists(), f"PreDist zip 파일이 없습니다: {ZIP_PATH}"
assert METADATA_DIR.exists(), f"metadata 폴더가 없습니다: {METADATA_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ZIP_PATH:", ZIP_PATH)
print("METADATA_DIR:", METADATA_DIR)

PROJECT_ROOT: C:\3rd_Project\HeatGridAgent
ZIP_PATH: C:\3rd_Project\HeatGridAgent\05_데이터셋\PreDist\predist_dataset.zip
METADATA_DIR: C:\3rd_Project\HeatGridAgent\05_데이터셋\PreDist\metadata


### 1.3 zip 내부 CSV 목록 확인

운영 시계열 CSV는 zip 안에 있고, metadata CSV는 별도 폴더에도 꺼내져 있다.

In [3]:
with ZipFile(ZIP_PATH) as zf:
    zip_csv_names = sorted(name for name in zf.namelist() if name.endswith(".csv"))

operational_paths = [name for name in zip_csv_names if "/operational_data/" in name]
m1_operational_paths = [name for name in operational_paths if name.startswith("manufacturer 1/")]
m2_operational_paths = [name for name in operational_paths if name.startswith("manufacturer 2/")]

zip_counts = {
    "전체 CSV": len(zip_csv_names),
    "운영 CSV": len(operational_paths),
    "manufacturer 1 운영 CSV": len(m1_operational_paths),
    "manufacturer 2 운영 CSV": len(m2_operational_paths),
}

zip_counts

{'전체 CSV': 101,
 '운영 CSV': 93,
 'manufacturer 1 운영 CSV': 35,
 'manufacturer 2 운영 CSV': 58}

In [4]:
expected_counts = {
    "전체 CSV": 101,
    "운영 CSV": 93,
    "manufacturer 1 운영 CSV": 35,
    "manufacturer 2 운영 CSV": 58,
}

assert zip_counts == expected_counts, f"예상 개수와 다릅니다: {zip_counts}"
pd.DataFrame(zip_counts.items(), columns=["구분", "개수"])

,구분,개수
0,전체 CSV,101
1,운영 CSV,93
2,manufacturer 1 운영 CSV,35
3,manufacturer 2 운영 CSV,58


## 2. 메타 데이터 기본 확인

### 2.1 metadata CSV 로드

`faults`, `disturbances`, `normal_events`, `feature_descriptions`를 제조사별 DataFrame으로 읽는다.

In [5]:
manufacturers = ["manufacturer_1", "manufacturer_2"]
metadata_files = ["faults", "disturbances", "normal_events", "feature_descriptions"]

metadata_frames = {}
for manufacturer in manufacturers:
    for file_type in metadata_files:
        csv_path = METADATA_DIR / manufacturer / f"{file_type}.csv"
        metadata_frames[(manufacturer, file_type)] = pd.read_csv(csv_path, sep=";")

list(metadata_frames.keys())

[('manufacturer_1', 'faults'),
 ('manufacturer_1', 'disturbances'),
 ('manufacturer_1', 'normal_events'),
 ('manufacturer_1', 'feature_descriptions'),
 ('manufacturer_2', 'faults'),
 ('manufacturer_2', 'disturbances'),
 ('manufacturer_2', 'normal_events'),
 ('manufacturer_2', 'feature_descriptions')]

### 2.2 메타 데이터 크기 확인

In [6]:
metadata_summary_rows = []
for (manufacturer, file_type), df in metadata_frames.items():
    metadata_summary_rows.append({
        "manufacturer": manufacturer,
        "file_type": file_type,
        "rows": len(df),
        "columns": len(df.columns),
    })

metadata_summary_df = pd.DataFrame(metadata_summary_rows)
metadata_summary_df

,manufacturer,file_type,rows,columns
0,manufacturer_1,faults,33,12
1,manufacturer_1,disturbances,165,3
2,manufacturer_1,normal_events,35,6
3,manufacturer_1,feature_descriptions,26,3
4,manufacturer_2,faults,40,12
5,manufacturer_2,disturbances,163,3
6,manufacturer_2,normal_events,30,6
7,manufacturer_2,feature_descriptions,48,2


In [7]:
expected_metadata_rows = {
    ("manufacturer_1", "faults"): 33,
    ("manufacturer_1", "disturbances"): 165,
    ("manufacturer_1", "normal_events"): 35,
    ("manufacturer_2", "faults"): 40,
    ("manufacturer_2", "disturbances"): 163,
    ("manufacturer_2", "normal_events"): 30,
}

actual_metadata_rows = {
    key: len(df)
    for key, df in metadata_frames.items()
    if key[1] in {"faults", "disturbances", "normal_events"}
}

assert actual_metadata_rows == expected_metadata_rows, actual_metadata_rows
actual_metadata_rows

{('manufacturer_1', 'faults'): 33,
 ('manufacturer_1', 'disturbances'): 165,
 ('manufacturer_1', 'normal_events'): 35,
 ('manufacturer_2', 'faults'): 40,
 ('manufacturer_2', 'disturbances'): 163,
 ('manufacturer_2', 'normal_events'): 30}

### 2.3 컬럼 타입과 결측치 확인

In [8]:
def describe_dataframe(name: str, df: pd.DataFrame) -> dict:
    return {
        "name": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }

basic_quality_df = pd.DataFrame([
    describe_dataframe(f"{manufacturer}/{file_type}", df)
    for (manufacturer, file_type), df in metadata_frames.items()
])
basic_quality_df

,name,rows,columns,missing_cells,duplicate_rows
0,manufacturer_1/faults,33,12,25,0
1,manufacturer_1/disturbances,165,3,0,0
2,manufacturer_1/normal_events,35,6,0,0
3,manufacturer_1/feature_descriptions,26,3,0,0
4,manufacturer_2/faults,40,12,62,0
5,manufacturer_2/disturbances,163,3,0,0
6,manufacturer_2/normal_events,30,6,0,0
7,manufacturer_2/feature_descriptions,48,2,11,0


In [9]:
for (manufacturer, file_type), df in metadata_frames.items():
    print(f"\n[{manufacturer} / {file_type}]")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.dtypes.rename("dtype").to_frame())
    display(df.isna().sum().rename("missing_count").to_frame())
    display(df.head(3))


[manufacturer_1 / faults]
shape: (33, 12)
columns: ['Event ID', 'substation ID', 'Report date', 'Problem EN', 'Event description EN', 'Possible anomaly start', 'Possible anomaly end', 'Training start', 'Training end', 'efd_possible', 'Fault label', 'Monitoring potential']


,dtype
Event ID,int64
substation ID,int64
Report date,str
Problem EN,str
Event description EN,str
Possible anomaly start,str
Possible anomaly end,str
Training start,str
Training end,str
efd_possible,bool


,missing_count
Event ID,0
substation ID,0
Report date,0
Problem EN,0
Event description EN,0
Possible anomaly start,16
Possible anomaly end,0
Training start,4
Training end,5
efd_possible,0


,Event ID,substation ID,Report date,Problem EN,Event description EN,Possible anomaly start,Possible anomaly end,Training start,Training end,efd_possible,Fault label,Monitoring potential
0,1,10,2014-05-04 14:44:00,no DHW,No hot water. Actuator (DHW system) replaced.,2014-05-03 16:00:00,2014-05-05 04:00:00,2012-03-28 09:00:00,2014-04-20 14:44:00,True,Motorised control valve (primary side): Actuat...,3.4
1,3,12,2015-12-01 10:56:00,no heat,Control parameters updated.,2015-11-29 12:00:00,2015-12-02 10:56:00,2015-03-01 00:00:00,2015-11-17 10:56:00,True,Control unit: Incorrect parameterisation,4
2,5,11,2018-11-23 08:30:00,no heat,Pump settings updated.,NaN,2018-11-26 09:56:59,2015-02-20 14:00:00,2018-11-09 08:30:00,True,Failure of the heating circuit pump,3.8



[manufacturer_1 / disturbances]
shape: (165, 3)
columns: ['substation ID', 'Event start', 'type']


...

,Event ID,substation ID,Event start,Event end,Training start,Training end
0,43,24,2020-02-20,2020-02-27,2019-10-16 08:40:00.310497,2020-02-20
1,44,53,2019-12-13,2019-12-20,2019-09-05 12:30:00.164418,2019-12-13
2,45,53,2020-02-18,2020-02-25,2019-09-05 12:30:00.164418,2020-02-18



[manufacturer_2 / feature_descriptions]
shape: (48, 2)
columns: ['column', 'unit']


,dtype
column,str
unit,str


,missing_count
column,0
unit,11


,column,unit
0,outdoor_temperature,°C
1,p_dhw_control_valve_position,%
2,p_dhw_return_temperature,°C


## 3. 운영 데이터 기본 확인

### 3.1 운영 CSV 인덱스 만들기

93개 운영 CSV를 바로 모두 DataFrame으로 올리지 않고, 먼저 목록 DataFrame으로 관리한다.

In [10]:
operational_index_rows = []
for zip_path in operational_paths:
    parts = zip_path.split("/")
    manufacturer = parts[0]
    file_name = parts[-1]
    substation_id = int(file_name.removeprefix("substation_").removesuffix(".csv"))
    operational_index_rows.append({
        "manufacturer": manufacturer,
        "substation_id": substation_id,
        "zip_path": zip_path,
    })

operational_index_df = pd.DataFrame(operational_index_rows).sort_values(
    ["manufacturer", "substation_id"]
).reset_index(drop=True)

operational_index_df

,manufacturer,substation_id,zip_path
0,manufacturer 1,1,manufacturer 1/operational_data/substation_1.csv
1,manufacturer 1,2,manufacturer 1/operational_data/substation_2.csv
2,manufacturer 1,3,manufacturer 1/operational_data/substation_3.csv
3,manufacturer 1,4,manufacturer 1/operational_data/substation_4.csv
4,manufacturer 1,5,manufacturer 1/operational_data/substation_5.csv
5,manufacturer 1,6,manufacturer 1/operational_data/substation_6.csv
6,manufacturer 1,7,manufacturer 1/operational_data/substation_7.csv
7,manufacturer 1,8,manufacturer 1/operational_data/substation_8.csv
8,manufacturer 1,9,manufacturer 1/operational_data/substation_9.csv
9,manufacturer 1,10,manufacturer 1/operational_data/substation_10.csv


### 3.2 운영 CSV 로드 함수

`manufacturer`는 `1` 또는 `2`, `substation_id`는 기계실 번호를 넣는다.

In [11]:
def load_operational_csv(manufacturer: int, substation_id: int) -> pd.DataFrame:
    zip_member = f"manufacturer {manufacturer}/operational_data/substation_{substation_id}.csv"
    with ZipFile(ZIP_PATH) as zf:
        if zip_member not in zf.namelist():
            raise FileNotFoundError(zip_member)
        with zf.open(zip_member) as raw_file:
            df = pd.read_csv(TextIOWrapper(raw_file, encoding="utf-8"), sep=";")

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    return df

sample_operational_frames = {
    "manufacturer_1_substation_1": load_operational_csv(1, 1),
    "manufacturer_2_substation_1": load_operational_csv(2, 1),
}

{name: df.shape for name, df in sample_operational_frames.items()}

{'manufacturer_1_substation_1': (112164, 15),
 'manufacturer_2_substation_1': (5094, 23)}

### 3.3 대표 운영 데이터 정보 확인

In [12]:
sample_summary_rows = []
for name, df in sample_operational_frames.items():
    timestamp_min = df["timestamp"].min() if "timestamp" in df.columns else pd.NaT
    timestamp_max = df["timestamp"].max() if "timestamp" in df.columns else pd.NaT
    sample_summary_rows.append({
        "sample": name,
        "rows": len(df),
        "columns": len(df.columns),
        "start_timestamp": timestamp_min,
        "end_timestamp": timestamp_max,
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

sample_operational_summary_df = pd.DataFrame(sample_summary_rows)
sample_operational_summary_df

,sample,rows,columns,start_timestamp,end_timestamp,missing_cells,duplicate_rows
0,manufacturer_1_substation_1,112164,15,2018-06-10 00:40:00,2020-07-28 23:50:00,55224,0
1,manufacturer_2_substation_1,5094,23,2020-02-26 15:10:00,2020-04-02 00:00:00,1618,0


In [13]:
for name, df in sample_operational_frames.items():
    print(f"\n[{name}]")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    display(df.dtypes.rename("dtype").to_frame())
    display((df.isna().mean() * 100).round(2).rename("missing_rate_percent").to_frame())
    display(df.head(5))


[manufacturer_1_substation_1]
shape: (112164, 15)
columns: ['timestamp', 'outdoor_temperature', 's_hc1_supply_temperature', 's_hc1_supply_temperature_setpoint', 's_dhw_supply_temperature', 's_dhw_supply_temperature_setpoint', 'p_hc1_return_temperature', 's_dhw_upper_storage_temperature', 's_dhw_lower_storage_temperature', 'p_net_meter_energy', 'p_net_meter_volume', 'p_net_meter_heat_power', 'p_net_meter_flow', 'p_net_supply_temperature', 'p_net_return_temperature']


,dtype
timestamp,datetime64[us]
outdoor_temperature,float64
s_hc1_supply_temperature,float64
s_hc1_supply_temperature_setpoint,float64
s_dhw_supply_temperature,float64
s_dhw_supply_temperature_setpoint,float64
p_hc1_return_temperature,float64
s_dhw_upper_storage_temperature,float64
s_dhw_lower_storage_temperature,float64
p_net_meter_energy,float64


,missing_rate_percent
timestamp,0.00
outdoor_temperature,0.00
s_hc1_supply_temperature,0.00
s_hc1_supply_temperature_setpoint,0.00
s_dhw_supply_temperature,0.00
s_dhw_supply_temperature_setpoint,0.00
p_hc1_return_temperature,0.00
s_dhw_upper_storage_temperature,0.00
s_dhw_lower_storage_temperature,0.00
p_net_meter_energy,8.21


,timestamp,outdoor_temperature,s_hc1_supply_temperature,s_hc1_supply_temperature_setpoint,s_dhw_supply_temperature,s_dhw_supply_temperature_setpoint,p_hc1_return_temperature,s_dhw_upper_storage_temperature,s_dhw_lower_storage_temperature,p_net_meter_energy,p_net_meter_volume,p_net_meter_heat_power,p_net_meter_flow,p_net_supply_temperature,p_net_return_temperature
0,2018-06-10 00:40:00,14.30,32.3,26.00,53.9,64.0,38.10,57.00,57.20,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-06-10 00:50:00,14.25,32.0,26.10,53.6,37.0,38.05,58.40,56.95,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-06-10 01:00:00,14.20,31.7,26.20,53.3,10.0,38.00,59.80,56.70,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-06-10 01:10:00,14.00,31.0,26.50,54.9,64.0,36.50,56.00,56.20,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-06-10 01:20:00,13.95,30.7,26.65,54.1,37.0,37.00,57.45,56.40,NaN,NaN,NaN,NaN,NaN,NaN



[manufacturer_2_substation_1]
shape: (5094, 23)
columns: ['timestamp', 's_dhw_supply_temperature_setpoint', 'outdoor_temperature', 's_hc1_control_unit_mode', 's_dhw_control_unit_mode', 's_hc1_room_temperature_setpoint', 'p_hc1_return_temperature_setpoint', 'p_hc1_return_temperature', 'p_dhw_return_temperature', 's_dhw_upper_storage_temperature', 's_dhw_lower_storage_temperature', 'p_hc1_control_valve_position_setpoint', 'p_dhw_control_valve_position', 's_hc1_heating_pump_status_setpoint', 's_hc1_supply_temperature_setpoint', 's_hc1_supply_temperature', 's_dhw_supply_temperature', 'p_net_meter_energy', 'p_net_meter_flow', 'p_net_meter_heat_power', 'p_net_return_temperature', 'p_net_meter_volume', 'p_net_supply_temperature']


,dtype
timestamp,datetime64[us]
s_dhw_supply_temperature_setpoint,float64
outdoor_temperature,float64
s_hc1_control_unit_mode,str
s_dhw_control_unit_mode,str
s_hc1_room_temperature_setpoint,float64
p_hc1_return_temperature_setpoint,float64
p_hc1_return_temperature,float64
p_dhw_return_temperature,float64
s_dhw_upper_storage_temperature,float64


,missing_rate_percent
timestamp,0.00
s_dhw_supply_temperature_setpoint,0.00
outdoor_temperature,0.00
s_hc1_control_unit_mode,0.00
s_dhw_control_unit_mode,0.00
s_hc1_room_temperature_setpoint,0.00
p_hc1_return_temperature_setpoint,0.00
p_hc1_return_temperature,0.00
p_dhw_return_temperature,0.00
s_dhw_upper_storage_temperature,0.00


,timestamp,s_dhw_supply_temperature_setpoint,outdoor_temperature,s_hc1_control_unit_mode,s_dhw_control_unit_mode,s_hc1_room_temperature_setpoint,p_hc1_return_temperature_setpoint,p_hc1_return_temperature,p_dhw_return_temperature,s_dhw_upper_storage_temperature,s_dhw_lower_storage_temperature,p_hc1_control_valve_position_setpoint,p_dhw_control_valve_position,s_hc1_heating_pump_status_setpoint,s_hc1_supply_temperature_setpoint,s_hc1_supply_temperature,s_dhw_supply_temperature,p_net_meter_energy,p_net_meter_flow,p_net_meter_heat_power,p_net_return_temperature,p_net_meter_volume,p_net_supply_temperature
0,2020-02-26 15:10:00,13.3,2.0,Tag,Tag,23.0,20.0,8.0,63.9,12.1,11.4,30.0,10.0,EIN,10.0,NaN,13.8,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-02-26 15:20:00,17.7,2.1,Tag,Tag,23.0,55.0,25.2,64.4,12.1,11.4,6.0,19.0,EIN,76.0,88.9,13.8,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-02-26 15:30:00,11.3,2.0,Tag,Tag,23.0,55.0,82.1,65.4,12.1,11.5,0.0,14.0,EIN,56.4,85.2,13.8,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-02-26 15:40:00,11.3,2.0,Tag,Tag,23.0,55.0,79.1,55.4,12.2,11.6,69.0,0.0,EIN,56.4,42.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-02-26 15:50:00,33.9,1.9,Tag,Tag,23.0,55.0,54.5,48.2,12.2,11.6,0.0,0.0,EIN,76.4,78.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. 선택 실행: 전체 운영 CSV 요약

아래 셀은 93개 운영 CSV를 하나씩 열어 행/열/기간만 요약한다. 전체 원본 데이터를 하나의 DataFrame으로 합치지는 않는다.

In [14]:
RUN_FULL_OPERATIONAL_PROFILE = False

if RUN_FULL_OPERATIONAL_PROFILE:
    profile_rows = []
    for row in operational_index_df.itertuples(index=False):
        manufacturer_number = int(row.manufacturer.replace("manufacturer ", ""))
        df = load_operational_csv(manufacturer_number, int(row.substation_id))
        profile_rows.append({
            "manufacturer": row.manufacturer,
            "substation_id": row.substation_id,
            "rows": len(df),
            "columns": len(df.columns),
            "start_timestamp": df["timestamp"].min() if "timestamp" in df.columns else pd.NaT,
            "end_timestamp": df["timestamp"].max() if "timestamp" in df.columns else pd.NaT,
        })
    full_operational_profile_df = pd.DataFrame(profile_rows)
    display(full_operational_profile_df)
else:
    print("기본 실행에서는 전체 운영 CSV 프로파일링을 건너뜁니다. 필요하면 RUN_FULL_OPERATIONAL_PROFILE = True로 바꾸세요.")

기본 실행에서는 전체 운영 CSV 프로파일링을 건너뜁니다. 필요하면 RUN_FULL_OPERATIONAL_PROFILE = True로 바꾸세요.


## 5. 확인 결과 메모

- PreDist는 운영 시계열만 있는 데이터가 아니라 `faults`, `disturbances`, `normal_events`, `feature_descriptions`가 함께 있는 라벨 포함 데이터셋이다.
- 운영 CSV는 제조사와 기계실마다 컬럼 구성이 다르므로 모델 입력을 만들기 전에 공통 컬럼, 결측치, timestamp 정렬 기준을 먼저 정해야 한다.
- 전체 운영 CSV를 하나로 합치기 전에 제조사별 컬럼 차이와 기계실별 기간 차이를 확인하는 것이 안전하다.